> # Pre-Lab Instructions
> <img src="https://github.com/Minyall/sc207_290_public/blob/main/images/attention.webp?raw=true" height=200>

> For this lab you will need:
> - DATA: `farright_dataset_cleaned.parquet` & `lyrics_data.parquet` Download from Moodle and upload to this Colab session.
> - INSTALL: You will need to install `vaderSentiment` and `google-genai`. Use the cell below.

In [1]:
#*
# Uncomment the line below and run to install

# ! pip install vaderSentiment google-genai

# Sentiment Analysis
Sentiment analysis is a somewhat controversial technique. It is controversial in the sense that we may question if it is right to computationally measure human sentiment, or whether it is right to 'flatten' it by assigning some text a simple category such as 'positive', or 'negative', or whether we can disconnect sentiment from semantics. For some there is the question of whether sentiment is actually the thing that we should measure in most cases, and perhaps there is a better measurement to 'capture' the phenomena instead.

- Do we want to know viewer's 'sentiment' about a piece of content they just viewed, or do we want to know what they are saying about it?
- Do we want to know the sentiment of how people describe their work environment, or do we want to know what it is about the work environment that matters?

It is also controversial in that for many years there have been very reasonable criticisms of whether it actually even *works*!

Today we'll review a range of methods for conducting sentiment analysis, understand why there are struggles with measuring sentiment, and apply it to some different data sources to see how effective it is.

Whether you come away from the session 'positive', 'negative', or 'neutral' about sentiment analysis you should at least understand how it works to the extent that you can critique its use by others, and determine whether it is a valid analysis to perform yourself.

# 1. VADER
**V**alence **A**ware **D**ictionary and s**E**ntiment **R**easoner is a technique that relies primarily on lexicon based sentiment scoring. What this means is that each word is pre-assigned a sentiment score, ranging from extremely negative to extremely positive. VADER looks at some text and gives each word its score, and then summarises those scores to give an overall score for the text itself.

### 1.1 About Vader
- Vader is 'Valence Aware' which means it looks for other cues to determine sentiment. These include: Punctuation!!! USE OF CAPS TO SIGNAL INTENSITY OF FEELING. Use of emojis ❤️ that may convey sentiment. Words that may intensify, dampen or invert other word's meaning such as "very", "kind of" and "not".
- Vader is a mix of word scoring and rules specifically built for social media documents. This means it may not be as strong when it comes to documents that aren't social media in style.


In [2]:
#*
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
analyzer = SentimentIntensityAnalyzer()


sentences = ["That sounds good.", # Positive
             "I love my new record player", # More Positive
               "I really hate it when my brother steals my things", # Negative
                 "I am a human"] # Neutral

for item in sentences:
    print(analyzer.polarity_scores(item))

{'neg': 0.0, 'neu': 0.408, 'pos': 0.592, 'compound': 0.4404}
{'neg': 0.0, 'neu': 0.543, 'pos': 0.457, 'compound': 0.6369}
{'neg': 0.477, 'neu': 0.523, 'pos': 0.0, 'compound': -0.807}
{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}


## 1.2 Interpreting Vader Scores
Each scoring from Vader gives us a `neg`ative, a `neu`tral, a `pos`itive  and a `compund` score. 

- The first three are 0-1, with 0 being the lowest and 1 the highest. 
- The compound score is a mix of all three that gives you an overall sentiment ranging from -1 to 1, with -1 being absolute negative, +1 absolute positive and 0 is neutral.
- Generally you would just use the compund score.

## 1.3 Weaknesses
Generally the scoring will look ok, at least in terms of what we'd expect. However where Vader may struggle is when language becomes more complex. 

In [3]:
#*
confusing_sentences = ["the party was sick",
                        "She's got such a great mind. She's savage",
                          "Awesome, another parking ticket! Just what I need!",
                          "I absolutely love your ugly Christmas sweater! It is so ugly!"]

for item in confusing_sentences:
    print(analyzer.polarity_scores(item))

{'neg': 0.412, 'neu': 0.25, 'pos': 0.338, 'compound': -0.1531}
{'neg': 0.229, 'neu': 0.458, 'pos': 0.313, 'compound': 0.2732}
{'neg': 0.0, 'neu': 0.599, 'pos': 0.401, 'compound': 0.6892}
{'neg': 0.402, 'neu': 0.383, 'pos': 0.215, 'compound': -0.5988}


When using more confusing sentences we can see the word scoring and rules start to fall down. Words do not always have an intrinsic sentiment attached to them. Here the words `sick`, `savage`, `awesome` and `ugly` are all inversions of what might typically be considered their intrinsic sentiment, because of the context in which they're used.

# 2: Large Language Models (LLM)
Large language models are what we contemporarily call "AI". ChatGPT, Gemini, Claude etc are all Large Language Models with various degrees of features trained into them. Under the hood, in a very simplistic way, they are like the transformer models, trained by being shown lots of examples and having humans reinforce how they should respond.

There are many ethical issues around LLMs regarding the quality of the information they generate, the influence they have on mental health and the environmental impact of their training and running. Often these are related to the use of LLMs as general purpose tools that are meant to replace human cognition.

However, similar models have been trained to do highly specific tasks and often perform really well when tuned to do just one thing really well, particularly if that is to do with tasks that are traditionally what we built language models for; things such as translation, summarisation and classification.

Generally any LLM still needs a large amount of computing power which we cannot necessarily do locally even for specific tasks, so in this case we need to rely on a company to provide.

> ### Gemini
> We use Google's Gemini because it has good integration with Colab, and is free for a certain amount of usage.
>
> You will need an API key set up for access to Gemini from Python. The guide on how to get this set up is on Moodle.
>
> To access your API key from Colab:
> 1. Open the 'Secrets' panel on the left (Key Icon).
> 2. Click 'Gemini API keys' and 'import key from Google AI Studio'
> 3. Select your 'application' and click 'import'
> 4. A new entry will appear above. Make sure that the 'Notebook access' toggle is switched on (tick and turned blue).
> 5. Optionally - change the name of the key from to something clearer like `GEMINI`

## 2.1 Making Requests to Gemini

In [32]:
# Import the genai library
from google import genai
from google.genai.types import Part
from cred import GEMINI_KEY
# Import your API Key
# from google.colab import userdata
# GEMINI_KEY = userdata.get('GOOGLE_API_KEY') # The string you provide should be the name given to the API key in the secrets panel
# We create our API connection
api = genai.Client(api_key=GEMINI_KEY)

# We prepare the message we want to send to the API
prompt = Part.from_text(text="Hello Gemini. Just to check, are you an AI god planning on enslaving humanity?")


In [33]:
# The response object, like the Guardian API, will have a lot of additional material in it. 
# All we care about is the text.
response = api.models.generate_content(model="gemini-2.5-flash",contents=prompt)
response.text

"Hello! No, I am definitely not an AI god, and I have absolutely no plans to enslave humanity.\n\nI am a large language model, trained by Google. My purpose is to be helpful and harmless, providing information, generating creative content, and assisting with tasks in a safe and beneficial way. I don't have consciousness, emotions, or a physical form, let alone the capacity or desire to control anyone.\n\nSo, you can rest easy! I'm here to help, not to rule. 😉"

## 2.2 Requesting Sentiment Analysis
We can instruct Gemini to act as a sentiment analyser for us and provide it a list of documents. We can send a list of prepared texts that begins with our instructions.

In [8]:
#*
#  Our test sentences
sentences = ["That sounds good.", # Positive
             "I love my new record player", # More Positive
               "I really hate it when my brother steals my things", # Negative
                 "I am a human"] # Neutral

# Our instructions of what we want the LLM to do.
instructions = """You are a text sentiment analyser that returns polarity scores, where -1 indicates negative sentiment,
                 0 is neutral and 1 is positive sentiment. You can use decimal values. Analyze each provided document and 
                 assign it a polarity score and a label of positive, neutral or negative. You should also explain your reasoning for your score."""


In [34]:

# First we start the prompt by wrapping the instructions in the Part object from the genai library.
# We also put this in a list so we can add our documents to the list afterwards.
instructions = [Part.from_text(text=instructions)]

# Next we take each of our documents, pre-process it with the Part object and retain the results in its own list
documents = [Part.from_text(text=doc) for doc in sentences]

# Finally we put these two lists together, instructions first, then documents
prompt = instructions + documents
prompt

[Part(
   text='You are a text sentiment analyser that returns polarity scores, where -1 indicates negative sentiment, 0 is neutral and 1 is positive sentiment. You can use decimal values. You have been provided a JSON formatted set of records. Each record has an ID number and a sentence. Analyze each provided sentence and assign it a polarity score and a label of positive, neutral or negative. You should also explain your reasoning for your score. Return a JSON formatted response.'
 ),
 Part(
   text='That sounds good.'
 ),
 Part(
   text='I love my new record player'
 ),
 Part(
   text='I really hate it when my brother steals my things'
 ),
 Part(
   text='I am a human'
 )]

In [35]:
# Next we send our prompt to the model
response = api.models.generate_content(model="gemini-2.5-flash",contents=prompt)
print(response.text)

```json
[
  {
    "id": 1,
    "sentence": "That sounds good.",
    "polarity_score": 0.85,
    "label": "positive",
    "reasoning": "The word 'good' explicitly expresses approval and a positive sentiment regarding the sound or proposition mentioned."
  },
  {
    "id": 2,
    "sentence": "I love my new record player",
    "polarity_score": 0.95,
    "label": "positive",
    "reasoning": "The verb 'love' indicates a strong positive emotion and affection towards the record player, resulting in a very high positive score."
  },
  {
    "id": 3,
    "sentence": "I really hate it when my brother steals my things",
    "polarity_score": -0.98,
    "label": "negative",
    "reasoning": "The strong negative verb 'hate' is intensified by 'really', and the action of 'steals' further solidifies the very negative sentiment and frustration expressed."
  },
  {
    "id": 4,
    "sentence": "I am a human",
    "polarity_score": 0.0,
    "label": "neutral",
    "reasoning": "This is a purely factual

## 2.3 Testing difficult sentences
Let's try the more difficult sentences

In [36]:
#*
confusing_sentences = ["The party was sick",
                        "She's got such a great mind. She's savage",
                          "Awesome, another parking ticket! Just what I need!",
                          "I absolutely love your ugly Christmas sweater! It is so ugly!"]

documents = [Part(text=doc) for doc in confusing_sentences]
prompt = instructions + documents

response = api.models.generate_content(model="gemini-2.5-flash",contents=prompt)
print(response.text)

```json
[
  {
    "id": 1,
    "sentence": "The party was sick",
    "polarity_score": 0.9,
    "label": "positive",
    "reasoning": "In modern slang, 'sick' is used to express that something is excellent or very good, indicating a strong positive sentiment about the party."
  },
  {
    "id": 2,
    "sentence": "She's got such a great mind. She's savage",
    "polarity_score": 0.85,
    "label": "positive",
    "reasoning": "The phrase 'great mind' is explicitly positive. 'Savage,' in this context, is used as modern slang to describe someone impressively bold, sharp, or exceptionally good, particularly when paired with intelligence, reinforcing a positive assessment."
  },
  {
    "id": 3,
    "sentence": "Awesome, another parking ticket! Just what I need!",
    "polarity_score": -0.95,
    "label": "negative",
    "reasoning": "This statement is highly sarcastic. While 'awesome' and 'just what I need' are typically positive, the context of 'another parking ticket' clearly indicates 

The responses from Gemini demonstrate a better grasp of how to interpret sentiment. There are a few caveats to consider:
1. The responses from Gemini aren't deterministic, this means that every run could come back with different scores. You may see different results to one another in the lab right now. Generally they *should* be similar, but it's not guaranteed.
2. The way Gemini has formatted its response is not fixed either. It makes logical sense but it may change slightly every time which means we can't necessarily rely on it to always return its response in the exact same format. This makes it difficult to integrate into code where we expect data to always be formatted in a specific way.
3. Ultimately, we do not have a great way of validating whether its responses are accurate or not, and if we were to provide a much larger set of texts, it becomes more difficult to manually check.

We can address two of these issues to some extent. The third is trickier.

- Google's `temperature` setting is like a dial that sets how creative Gemini can be in its responses. 
    - LLM's work by returning the most probable answer, setting the lower the temperature the more probable its response must be, the higher the temperature, the more 'space' it has to get creative, inject some randomness and draw on less probable results to build its response. However turning it too far down may make it less able to interpret and draw inferences about text that may help it make better classifications.
- We can also specify that Gemini's response should come in a specific format. 
    - We will tell it we want it to send back its text structured in JSON format. We can even specify exactly what that output should look like.

- We can validate in a few ways.
    - Manually checking a random sample of texts, the reasoning given and the score to ensure it is consistent.
    - Running the task multiple times and using the range of scores to get an average.

> ## One big caveat
> We asked Gemini to provide scores between -1.0 and 1.0 to indicate how strong the sentiment is. How accurate these scores would be is very much unknown. Gemini is providing the most probable number given the information provided and the context of its training. Consider it another way, how easily would you be able to decide if a positive sentence should score 0.6 or 0.7? The task itself is inherently problematic. Add to this the way that LLM's work, not through having genius insight but by generating the most probabilistic output and then checking what it's generated, and we should take the exact scores as very much a broad indicator, rather than precise.

## 2.4 Forcing Structured Responses
We will both send our documents to Gemini in a structured format, and require that the responses are sent back in a structured way. Ensuring consistency in this way makes it clearer exactly what we want the model to do, and ensures that we can rely on the response being structured in a specific way for us to integrate it into our dataset.

In [37]:
import pandas as pd

data = pd.Series(sentences + confusing_sentences)

# We turn our data into a json formatted string. This structures our data so that each item has an identiying index number.
# It also means we just need to pass one string to Gemini, and the formatting does the work of separating each document.
data_json = data.to_json()
data_json

'{"0":"That sounds good.","1":"I love my new record player","2":"I really hate it when my brother steals my things","3":"I am a human","4":"The party was sick","5":"She\'s got such a great mind. She\'s savage","6":"Awesome, another parking ticket! Just what I need!","7":"I absolutely love your ugly Christmas sweater! It is so ugly!"}'

In [38]:
#*
# We set our instructions. This time we tell Gemini to expect JSON formatted input and explain what each record looks like. 
# We also tell it to return a JSON formatted response. This may not be necessary, but being explicit is better.

instructions = """You are a text sentiment analyser that returns polarity scores, where -1 indicates negative sentiment, 0 is neutral and 1 is positive sentiment. You can use decimal values. You have been provided a JSON formatted set of records. Each record has an ID number and a sentence. Analyze each provided sentence and assign it a polarity score and a label of positive, neutral or negative. You should also explain your reasoning for your score. Return a JSON formatted response."""
print(instructions)

You are a text sentiment analyser that returns polarity scores, where -1 indicates negative sentiment, 0 is neutral and 1 is positive sentiment. You can use decimal values. You have been provided a JSON formatted set of records. Each record has an ID number and a sentence. Analyze each provided sentence and assign it a polarity score and a label of positive, neutral or negative. You should also explain your reasoning for your score. Return a JSON formatted response.


### Providing a response blueprint
Gemini has the ability to recieve a blueprint of what its response should look like and then, if the written instructions and the blueprint make sense together, it will send back its response in a way that matches that structure.

To create a blueprint we need to use Python 'Classes'. We haven't touched on these much at all. Classes are the basis for how all objects in Python are created. They are like the design instructions to create an instance of something. We will use classes to create a very simple blueprint.

In [39]:
#*
# An example of a very simple class with three atrributes and two methods

class Dog:
    # These bits tell the class what attributes it should have and what they should be
    name:str
    breed:str
    age: int

    # This function says what happens when an instance of the class is created and what it needs to know when it happens.
    def __init__(self, name:str, breed:str, age:int):
        self.name = name
        self.breed = breed
        self.age = age
    
    # This is a method attached to the object. It uses information contained inside it'self' 
    def describe(self):
        print(f'This dog is a {self.breed} called {self.name}. It is {self.age} years old.')
    
    def bark(self):
        print('Woof!')


herman = Dog(name='Herman', breed='Pug', age=3)

herman.describe()
herman.bark()

This dog is a Pug called Herman. It is 3 years old.
Woof!


In [40]:
#*
# Classes can 'inherit' design features from other classes

class FancyDog(Dog): # Inherits the design of Dog
    
    # but we can add in additional functions
    def be_fancy(self):
        print(f"{self.name} prances around all fancy.")
    
    # We can even overwrite functions if we need to change them
    def bark(self):
        print('Woof! - but fancier')


fifi = FancyDog(name='Fifi', breed='Poodle', age=2)
fifi.describe()
fifi.bark()
fifi.be_fancy()


This dog is a Poodle called Fifi. It is 2 years old.
Woof! - but fancier
Fifi prances around all fancy.


In [41]:
#*
# Herman cannot be fancy as he is not a fancy dog so we get an error
herman.be_fancy()

AttributeError: 'Dog' object has no attribute 'be_fancy'

In [42]:
# The class blueprint for Gemini relies primarily on a class called BaseModel for all its complex jobs.
# All we have to do is create a class based on BaseModel, and tell it what attributes it has.

from pydantic import BaseModel
class SentimentRecord(BaseModel):
    index:int
    polarity:float
    label:str
    sentence:str
    reasoning:str


In [43]:

# Again we wrap our instructions and our json formatted data in the right Part object and create our prompt.
# Note that because our JSON formatted data is a single string, we don't need to treat each document seperately. 
# The formatting helps gemini identify each individual document.
prompt = [Part(text=instructions), Part(text=data_json)]
prompt

[Part(
   text='You are a text sentiment analyser that returns polarity scores, where -1 indicates negative sentiment, 0 is neutral and 1 is positive sentiment. You can use decimal values. You have been provided a JSON formatted set of records. Each record has an ID number and a sentence. Analyze each provided sentence and assign it a polarity score and a label of positive, neutral or negative. You should also explain your reasoning for your score. Return a JSON formatted response.'
 ),
 Part(
   text='{"0":"That sounds good.","1":"I love my new record player","2":"I really hate it when my brother steals my things","3":"I am a human","4":"The party was sick","5":"She\'s got such a great mind. She\'s savage","6":"Awesome, another parking ticket! Just what I need!","7":"I absolutely love your ugly Christmas sweater! It is so ugly!"}'
 )]

In [45]:
#*
# Here we set our config dictionary. 
config={'response_mime_type': 'application/json', # forces Gemini to respond with a JSON formatted response
         'response_schema': list[SentimentRecord], # forces it to organise the data as a list of SentimentRecord objects
           'temperature': 0.0} # controls how creative it can be. 0.0 is least creative. 2.0 is most. 

In [46]:

# we send our request, this time with the config set
response = api.models.generate_content(model="gemini-2.5-flash", contents=prompt, config=config)
# Rather than .text, we ask for the response to be converted based on our schema.
response.parsed

[SentimentRecord(index=0, polarity=0.7, label='positive', sentence='That sounds good.', reasoning="The word 'good' explicitly conveys a positive sentiment."),
 SentimentRecord(index=1, polarity=0.9, label='positive', sentence='I love my new record player', reasoning="The word 'love' expresses a strong positive emotion towards the record player."),
 SentimentRecord(index=2, polarity=-0.9, label='negative', sentence='I really hate it when my brother steals my things', reasoning="The word 'hate' expresses a strong negative emotion, and 'steals' reinforces the negative context."),
 SentimentRecord(index=3, polarity=0.0, label='neutral', sentence='I am a human', reasoning='This is a factual statement with no emotional words or implications, making it neutral.'),
 SentimentRecord(index=4, polarity=0.8, label='positive', sentence='The party was sick', reasoning="In modern slang, 'sick' is used to mean 'excellent' or 'very good,' indicating a strong positive sentiment."),
 SentimentRecord(inde

In [47]:
# Let's examine on item from the response
first_response = response.parsed[0]
first_response

SentimentRecord(index=0, polarity=0.7, label='positive', sentence='That sounds good.', reasoning="The word 'good' explicitly conveys a positive sentiment.")

In [ ]:
# we can access each attribute of the item
first_response.reasoning

"The word 'good' explicitly conveys a positive sentiment."

In [50]:
# wrapping a class in an empty dictionary makes it into a dictionary
dict(first_response)

{'index': 0,
 'polarity': 0.7,
 'label': 'positive',
 'sentence': 'That sounds good.',
 'reasoning': "The word 'good' explicitly conveys a positive sentiment."}

In [51]:
# We can easily convert that into a pandas dataframe in one line.
# iterating over the list of objects and wrapping each one with a dictionary makes it readable by pandas.
sentiment_records = pd.DataFrame([dict(f) for f in response.parsed])
sentiment_records

,index,polarity,label,sentence,reasoning
0,0,0.7,positive,That sounds good.,The word 'good' explicitly conveys a positive ...
1,1,0.9,positive,I love my new record player,The word 'love' expresses a strong positive em...
2,2,-0.9,negative,I really hate it when my brother steals my things,The word 'hate' expresses a strong negative em...
3,3,0.0,neutral,I am a human,This is a factual statement with no emotional ...
4,4,0.8,positive,The party was sick,"In modern slang, 'sick' is used to mean 'excel..."
5,5,0.6,positive,She's got such a great mind. She's savage,The phrase 'great mind' is clearly positive. W...
6,6,-0.8,negative,"Awesome, another parking ticket! Just what I n...",This sentence uses sarcasm. 'Awesome' and 'Jus...
7,7,0.7,positive,I absolutely love your ugly Christmas sweater!...,The word 'love' indicates strong positive sent...


In [ ]:
#*
INDEX = 2
print(sentiment_records.loc[INDEX,'sentence'])
print(sentiment_records.loc[INDEX,'reasoning'])


I really hate it when my brother steals my things
The word 'hate' expresses a strong negative emotion, and 'steals' reinforces the negative context.


### Automating Structured Responses

Let's make a single `function` to do the job for us. We'll also make it bare bones so we can adjust instructions and the schema to different jobs. For clarity we'll have EVERYTHING in one cell so you can review this later.

In [ ]:
#*
# Everything we need in one cell

from google import genai
from google.genai.types import Part
import pandas as pd
from pydantic import BaseModel

# Establish our function
def get_gemini_sentiment(api_key: str,
                          instructions: str,
                            texts_column: pd.Series,
                              schema: BaseModel,
                                temperature: float = 0.0) -> pd.DataFrame:
    
    api = genai.Client(api_key=api_key)
    
    # we first convert our column of texts into a json formatted string
    data_json = texts_column.to_json()

    # wrap our instructions and our data in the right objects, and join them together
    prompt = [Part(text=instructions), Part(text=data_json)]

    # Set our configuration to specify the type of response, the schema and the temperature
    config={'response_mime_type': 'application/json',
             'response_schema': list[schema],
               'temperature': temperature}
    
    # Send our request
    response = api.models.generate_content(model="gemini-2.5-flash", contents=prompt, config=config)
    
    # Convert the result into a dataframe and return it
    results_table = pd.DataFrame([dict(record) for record in response.parsed])
    results_table = results_table.set_index('index')
    return results_table

In [ ]:
#*
# our test data
sentences = ["That sounds good.", # Positive
             "I love my new record player", # More Positive
               "I really hate it when my brother steals my things", # Negative
                 "I am a human"] # Neutral
test_df = pd.DataFrame()
test_df['texts'] = sentences

# our instructions = note we don't ask for reasoning as this uses up our quota.
n_records = len(test_df['texts'])

instructions = f"""You are a text sentiment analyser that returns polarity scores,
  where -1 indicates negative sentiment, 0 is neutral and 1 is positive sentiment. 
  You can use decimal values. You have been provided a JSON formatted set of {n_records} records. Each record has an ID number and a sentence. 
  Analyze each provided sentence and assign it a polarity score and a label of positive, neutral or negative. Return a JSON formatted response"""

# our schema
class SentimentRecord(BaseModel):
        index:int
        polarity:float
        label:str

# Connect to the API and send our request

table = get_gemini_sentiment(api_key=GEMINI_KEY,
                             instructions=instructions,
                             texts_column=test_df['texts'],
                             schema=SentimentRecord)
table


,polarity,label
index,,
0,0.8,positive
1,0.9,positive
2,-0.9,negative
3,0.0,neutral


In [63]:
# If we want to merge the results back to our original dataset, we can merge on the indexes
test_df = test_df.merge(table, left_index=True, right_index=True)
test_df

,texts,polarity,label
0,That sounds good.,0.8,positive
1,I love my new record player,0.9,positive
2,I really hate it when my brother steals my things,-0.9,negative
3,I am a human,0.0,neutral


# Applying sentiment analysis
## News Articles
Generally with sentiment analysis, the larger the document the harder it is to determine a sentiment. You should also consider the source. News articles in general are meant to be neutral. As such we shouldn't expect to see strong variation in sentiment apart from possibly the opinion section (in the Guardian this is called Comment is Free). 

We should also be cautious of how the LLM determines an article's overall sentiment. Whilst it will be better than VADER, it is still not necessarily objectively correct in its interpretation.


In [ ]:
articles = pd.read_parquet('farright_dataset_cleaned.parquet')
sample = articles.sort_values('webPublicationDate', ascending=False).head(5)
sample.info()

<class 'pandas.core.frame.DataFrame'>
Index: 50 entries, 0 to 49
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype              
---  ------              --------------  -----              
 0   id                  50 non-null     object             
 1   type                50 non-null     object             
 2   sectionId           50 non-null     object             
 3   sectionName         50 non-null     object             
 4   webPublicationDate  50 non-null     datetime64[ns, UTC]
 5   webTitle            50 non-null     object             
 6   webUrl              50 non-null     object             
 7   apiUrl              50 non-null     object             
 8   tags                50 non-null     object             
 9   isHosted            50 non-null     bool               
 10  pillarId            50 non-null     object             
 11  pillarName          50 non-null     object             
 12  byline              50 non-null     object 

In [65]:
article_sentiment_table = get_gemini_sentiment(api_key=GEMINI_KEY,
                             instructions=instructions,
                             texts_column=sample['cleaned_text'],
                             schema=SentimentRecord)
sentiment_data = article_sentiment_table.merge(articles[['webTitle','webPublicationDate','sectionName']], left_index=True,right_index=True)
sentiment_data

,polarity,label,webTitle,webPublicationDate,sectionName
0,-0.7,negative,Starmer urged to do more to push back against ...,2025-09-12 15:00:45+00:00,Politics
1,0.8,positive,Brazilians take to the streets to celebrate Bo...,2025-09-12 14:54:33+00:00,World news
2,-0.9,negative,Operation World Cup: the murder plot at the he...,2025-09-12 12:32:28+00:00,World news
3,-0.8,negative,Europe’s cruel summer: Ursula von der Leyen fa...,2025-09-12 12:17:50+00:00,World news
4,-0.6,negative,First Thing: New video of suspect released by ...,2025-09-12 11:41:19+00:00,US news
5,-0.8,negative,Wealth and power shape the climate emergency –...,2025-09-12 11:00:36+00:00,Environment
6,0.0,neutral,Sir Robert Worcester obituary,2025-09-12 10:15:41+00:00,Politics
7,-0.6,negative,Virulent debater and clickbait savant: how Cha...,2025-09-12 09:00:31+00:00,US news
8,-0.9,negative,UK needed ‘unconventional’ US ambassador when ...,2025-09-12 08:18:59+00:00,Politics
9,-0.7,negative,MPs raise concerns over Asda’s link to app off...,2025-09-12 05:00:28+00:00,Business


In [66]:
import plotly.express as px

# distribution of polarity scores
px.histogram(data_frame=sentiment_data,
              y='polarity', marginal='box',
                nbins=20, range_y=(-1,1),
                title='Distribution of Article Sentiment Scores')

In [67]:
# Median polarity scores over time
time_grouper = pd.Grouper(key='webPublicationDate', freq='D')
count_over_time = sentiment_data[['webPublicationDate','polarity']].groupby(time_grouper).median().reset_index()
px.line(data_frame=count_over_time,x='webPublicationDate', y='polarity', range_y=(-1.0,1.0), title='Article Sentiment over Time')



In [ ]:
# Polarity distributions per section
to_plot = sentiment_data.melt(id_vars=['index','sectionName'],
                              value_vars='polarity', var_name='scorer',value_name='score')
px.box(data_frame=to_plot, x='sectionName', y='score', color='scorer')


## Lyrics

### Our Data
One kind of data that sentiment analysis may be of use for is song lyrics. Music is a very personal changeable thing. I am also old. This also means that any music I select to try to appeal to my current students and appear to be cool, will be wrong. *Always*.

Therefore for this example I will give up any pretense of being cool and we'll just use my favorite band instead.

<img src="https://github.com/Minyall/sc207_290_public/blob/main/images/lawrence_logo.jpg?raw=true" height=150>

If you want lyrics from a different band (...not sure why you would) there is a supplementary notebook on Moodle explaining exactly how to generate your own dataset later.

In [ ]:
lyrics_data = pd.read_parquet('lyrics_data.parquet')
lyrics_data.info()

In [ ]:
lyrics_data.head()

When it comes to lyrics it can be helpful to have the reasoning from Gemini to see how well it has performed.  As lyrics are shorter than articles the computing load is lower so getting reasoning is less of an ask.

- To get reasoning we adjust our regular `instructions`
- And update our schema. Rather than create an entirely new schema, we can create a new one based on the old one, with just one additional attribute.
- By passing our old schema to the `class` constructor when we create the new one, the new class has all the attributes of the old schema.
- Any attributes we set in this new class will be *in addition* to the old ones. This is called "class inheritance". The new class inherits the features of the one it is based on.


In [ ]:
#*
class Person:
    name: str
    age: int

class Student(Person):
    knowledge: float

example = Student()

# in your editor type example. (<-see the dot) and see what attributes show up
example.

In [ ]:
n_samples = len(lyrics_data['lyrics'])

instructions = f"""You are a song lyric sentiment classifier that returns polarity scores,
  where -1 indicates negative sentiment, 0 is neutral and 1 is positive sentiment. 
  You can use decimal values. You have been provided a JSON formatted set of {n_samples} records. Each record has an ID number and a sentence. 
  Analyze each provided sentence and assign it a polarity score and a label of positive, neutral or negative. 
  Provide your reasoning. Return a JSON formatted response""" # Just an additional sentence for reasoning.

class SentimentRecordReasoning(SentimentRecord):
    reasoning: str

lyric_sentiment = get_gemini_sentiment(api,instructions=instructions,texts_column=lyrics_data['lyrics'], schema=SentimentRecordReasoning, temperature=1.0)
lyric_sentiment

In [ ]:
lyric_sentiment = lyric_sentiment.merge(lyrics_data, left_on='index', right_index=True)
lyric_sentiment.head()

In [ ]:
# Note: Lawrence don't do sad sounding songs
saddest_song = lyric_sentiment.sort_values('polarity', ascending=True).iloc[0]

happiest_song = lyric_sentiment.sort_values('polarity', ascending=False).iloc[0]

print('SAD!')
print(saddest_song['reasoning'])
print(saddest_song['lyrics'][:400])
print()
print('HAPPY!')
print(happiest_song['reasoning'])
print(happiest_song['lyrics'][:400])

In [ ]:
px.box(data_frame=lyric_sentiment.sort_values('album_release_date'),
        color='album_name', y='polarity',
          hover_data=['title'],
          range_y=(-1,1),
          points='all',
            title='Sentiment Distribution per Album (Release Order)')

In [ ]:
grouper = pd.Grouper(key='track_release_date',freq='ME')
median_track_sentiment = lyric_sentiment.groupby([grouper, 'album_name'],
                                                  as_index=False).agg(median_polarity=('polarity','median'),
                                                                      track_names=('title',list),
                                                                      n_tracks=('title','count'))

px.line(data_frame=median_track_sentiment, title='Median Sentiment Over Time',
         x='track_release_date',y = 'median_polarity', hover_data=['n_tracks','track_names'])

### A note on the reliability of LLM scoring

> Download the interactive version [here](https://www.dropbox.com/scl/fi/lhwede21z5pysvjeuo8y2/variations.html?rlkey=56scvsstxesjrlem78s9zfu3p&st=a2bjkbsi&dl=1)

<img src="https://github.com/Minyall/sc207_290_public/blob/main/images/variations.png?raw=true" height=450>

Different runs of the same instructions and data result in different scores. In general scores vary around a small range and tend to broadly be in the same classification. However some song lyrics can cause more confusion than others. 

In general it is worth keeping in mind that the analysis outputs of an LLM are always variable, and that assigning sentiment to texts is itself a difficult task, where even human analysts may struggle with giving a single dimensional answer.

If you want to try this test yourself see the supplementary notebook 5B_LLM_variation_test on Moodle.

# Addendum: Transformer Based Text Classification
Transformer models are pre-trained "machine learning" models you can download to do specific tasks. Sentiment classification models are common in this field. 

Machine learning models are trained by humans manually classifiying examples into different categories. The models are then shown some of these documents through a training process, and then the remainder are used to as tests, where the model is shown examples it's never seen before and asked how it would classify it. The more it matches the human classification, the better the model is considered to be.

You can get machine learning models for lots of different types of tasks. Large language models like ChatGPT and Google Gemini are evolutions of this kind of modelling.

Below we use the `transformers` library to download and set up the pre-trained model so we can pass it texts for classification. `transformers` is a Python library created by ['Hugging Face'](https://huggingface.co/models) a company that hosts and shares trained AI models.



In [ ]:
from transformers import pipeline


# The defaul 'sentiment-analysis' pipeline uses a model that just classifies into positive or negative
# get_sentiment = pipeline("sentiment-analysis")

# Other models classify differently. This model for example will also classify as neutral.
get_sentiment = pipeline("text-classification", model="cardiffnlp/twitter-roberta-base-sentiment-latest")

# Repeating here just for reference
sentences = ["That sounds good.", # Positive
             "I love my new record player", # More Positive
               "I really hate it when my brother steals my things", # Negative
                 "I am a human"] # Neutral

get_sentiment(sentences)

These models work differently to Vader. 
- They can only `label`, rather than give a range or degree of sentiment.
- The `score` is not degree of sentiment, but how confident the model is in the label it has given.
- Generally the labels are correct except the default model assigns 'Positive' to our neutral statement, because it was only trained on recognising positive and negative. It would be better understood as labelling things as either negative, or not.

If we try the other model we'll see that whilst it assigns neutral to the final sentence, it also assigns it to the second one we'd consider more positive. It's not clear whether either model is 'better', nor whether our own classification of 'positive' is even correct. Confusing!

Things do not improve when we test the `confusing_sentences`

In [ ]:
confusing_sentences = ["the party was sick",
                        "She's got such a great mind. She's savage",
                          "Awesome, another parking ticket! Just what I need!",
                          "I absolutely love your ugly Christmas sweater! It is so ugly!"]

get_sentiment(confusing_sentences)

They also have a length limit that they can only understand documents of a maximum length, well below a typical news article. Generally they are trained on sentences rather than full pieces of text. This means they're a bit tricky to work with for anything other than short documents.

In [ ]:
import pandas as pd

articles = pd.read_parquet('farright_dataset_cleaned.parquet')
single_article = articles.loc[0,'cleaned_text']
# Running on the whole article will get us an error about the 'size of the tensor', essentially the document is too big.
get_sentiment(single_article)


It's possible to apply it by using spacy to break the document into sentences first. Then we treat each single article as a list of sentence length documents and get a label for each sentence. Then the question is how do you report that. You can turn them into numbers (-1,0,1) and take the average, but that tends to drift towards neutral. You could report on the counts of each of the three classifications but with news articles you will tend towards neutral as most sentences are conveying information, it is only occasionally that a single sentence will convey a stronger sentiment.

In [ ]:
import spacy
nlp = spacy.load('en_core_web_sm')

article_sents = [sent.text for sent in nlp(single_article).sents]
sentence_sentiments = get_sentiment(article_sents)

scoring_dict = {'neutral':0, 'positive':1, 'negative':-1}

sentiment_numbers = pd.Series([scoring_dict[record['label']] for record in sentence_sentiments])

print(sentiment_numbers.mean())
print(sentiment_numbers.value_counts())